In [102]:
import numpy as np
import pandas as pd
import sys
sys.path.append("C:/Users/wg984/Wolfgang/repos/Bedmaster-ICU-tools/code/")
from bmresearch_tools import BMR_load, BMR_plot, get_metadata
import datetime
import h5py
import os
%matplotlib widget
import matplotlib.pyplot as plt

In [5]:
bmr_studyid_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_studyID/'
ecg_studyid_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG/'
bm_files = os.listdir(bmr_studyid_dir)
bm_file = bm_files[0]
bm_filepath = os.path.join(bmr_studyid_dir, bm_file)
output_h5_path = os.path.join(ecg_studyid_dir, bm_file.replace('BMR','ECG'))

In [9]:
signals_in_bm_file = get_metadata(bm_filepath)
waveform_signals = ['I','II','III', 'IIII','V','VI','VII','VIII','SPO2','CO2','ART1','ART2','RESP']
waveform_signals = [x for x in waveform_signals if x in signals_in_bm_file]
print(waveform_signals)
new_fs = 240

['I', 'II', 'III', 'V', 'SPO2', 'ART1', 'RESP']


In [10]:
waveform_signals = ['I','SPO2']
# load the data:
data = BMR_load(bm_filepath, signals = waveform_signals, loadEvents=0)

In [198]:
for signal in data.keys():
#     data[signal].set_index('datetime',inplace=True)
#     data[signal].drop('posix', axis=1, inplace=True)
    data[signal].columns = [signal]

In [205]:
data2 = [data[x] for x in data]

In [209]:
data2

[                              I
 datetime                       
 2018-06-05 07:17:34.000000  2.0
 2018-06-05 07:17:34.004167  4.0
 2018-06-05 07:17:34.008333  7.0
 2018-06-05 07:17:34.012500  7.0
 2018-06-05 07:17:34.016667  3.0
 ...                         ...
 2018-06-08 14:32:47.445833  0.0
 2018-06-08 14:32:47.450000  4.0
 2018-06-08 14:32:47.454167  5.0
 2018-06-08 14:32:47.458333  9.0
 2018-06-08 14:32:47.462500  5.0
 
 [65106936 rows x 1 columns],                               SPO2
 datetime                          
 2018-06-05 07:17:34.000000   648.0
 2018-06-05 07:17:34.016667   720.0
 2018-06-05 07:17:34.033333   808.0
 2018-06-05 07:17:34.050000   912.0
 2018-06-05 07:17:34.066667  1024.0
 ...                            ...
 2018-06-08 14:32:47.383333  1520.0
 2018-06-08 14:32:47.400000  1528.0
 2018-06-08 14:32:47.416667  1544.0
 2018-06-08 14:32:47.433333  1560.0
 2018-06-08 14:32:47.450000  1568.0
 
 [15945771 rows x 1 columns]]

In [233]:
data3 = [data2[x].iloc[:10000] for x in range(len(data2))]

In [234]:
data3

[                              I
 datetime                       
 2018-06-05 07:17:34.000000  2.0
 2018-06-05 07:17:34.004167  4.0
 2018-06-05 07:17:34.008333  7.0
 2018-06-05 07:17:34.012500  7.0
 2018-06-05 07:17:34.016667  3.0
 ...                         ...
 2018-06-05 07:18:15.645833  7.0
 2018-06-05 07:18:15.650000  6.0
 2018-06-05 07:18:15.654167  4.0
 2018-06-05 07:18:15.658333  4.0
 2018-06-05 07:18:15.662500  5.0
 
 [10000 rows x 1 columns],                               SPO2
 datetime                          
 2018-06-05 07:17:34.000000   648.0
 2018-06-05 07:17:34.016667   720.0
 2018-06-05 07:17:34.033333   808.0
 2018-06-05 07:17:34.050000   912.0
 2018-06-05 07:17:34.066667  1024.0
 ...                            ...
 2018-06-05 07:20:20.583333   960.0
 2018-06-05 07:20:20.600000   944.0
 2018-06-05 07:20:20.616667   920.0
 2018-06-05 07:20:20.633333   896.0
 2018-06-05 07:20:20.650000   864.0
 
 [10000 rows x 1 columns]]

In [235]:
merge = pd.concat(data3,join='outer', axis=1,sort=True)

In [236]:
merge

,I,SPO2
datetime,,
2018-06-05 07:17:34.000000,2.0,648.0
2018-06-05 07:17:34.004167,4.0,NaN
2018-06-05 07:17:34.008333,7.0,NaN
2018-06-05 07:17:34.012500,7.0,NaN
2018-06-05 07:17:34.016667,3.0,720.0
...,...,...
2018-06-05 07:20:20.583333,NaN,960.0
2018-06-05 07:20:20.600000,NaN,944.0
2018-06-05 07:20:20.616667,NaN,920.0


In [237]:
merge_data_original_start = merge.index[0]
merge_data_original_start

Timestamp('2018-06-05 07:17:34')

In [238]:
new_fs = 240
merge = merge.resample(datetime.timedelta(seconds = 1/new_fs)).mean() 


In [239]:
merge

,I,SPO2
datetime,,
2018-06-05 07:17:33.995985,2.0,648.0
2018-06-05 07:17:34.000152,4.0,NaN
2018-06-05 07:17:34.004319,7.0,NaN
2018-06-05 07:17:34.008486,7.0,NaN
2018-06-05 07:17:34.012653,3.0,720.0
...,...,...
2018-06-05 07:20:20.630148,NaN,896.0
2018-06-05 07:20:20.634315,NaN,NaN
2018-06-05 07:20:20.638482,NaN,NaN


In [240]:
merge = merge.interpolate(method='pchip', order =3, limit_area ='inside', limit = 25)   #  limit = 4. limit can be used to not interpolate gaps. however, matlab cardiovascular toolbox does expect continuous data. therefore interpolate also gaps.

In [241]:
merge

,I,SPO2
datetime,,
2018-06-05 07:17:33.995985,2.0,648.000000
2018-06-05 07:17:34.000152,4.0,664.537894
2018-06-05 07:17:34.004319,7.0,682.099837
2018-06-05 07:17:34.008486,7.0,700.611803
2018-06-05 07:17:34.012653,3.0,720.000000
...,...,...
2018-06-05 07:20:20.630148,NaN,896.000000
2018-06-05 07:20:20.634315,NaN,888.830512
2018-06-05 07:20:20.638482,NaN,881.071723


In [242]:
plt.close()
# plt.plot(sub.index, sub.signal_I)
# plt.plot(sub2.index, sub2.signal_SPO2)
plt.plot(merge.index, merge.I)
plt.plot(merge.index, merge.SPO2)
# plt.ylim([-500,3000])
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [ ]:
# save as h5 file

In [ ]:
### end of code

In [16]:
waveform_signals = ['I','II','III','SPO2','ART1']
ff = h5py.File(bm_filepath, 'r')
start_posix = []
for signal in waveform_signals:
    start_posix.append(ff[signal + '_dt'][0])
min(start_posix)

1528197454.0

In [ ]:
ff['_dt'][0]

In [ ]:
data = lead_data[lead]

In [ ]:
# resample to 240Hz:
lead_data.set_index('datetime', inplace=True)
lead_data = lead_data[['signal']]
lead_data_original_start = lead_data.index[0]
lead_data = lead_data.resample(datetime.timedelta(seconds = 1/new_fs)).mean() 
# lead_data_resampled_start = lead_data.index[0]
# difference_start = lead_data_original_start - lead_data_resampled_start
lead_data = lead_data.interpolate(method='pchip', order =3)   #  limit = 4. limit can be used to not interpolate gaps. however, matlab cardiovascular toolbox does expect continuous data. therefore interpolate also gaps.
# lead_data.index = lead_data.index + difference_start
datetime_array = np.array([lead_data_original_start.year, lead_data_original_start.month, lead_data_original_start.day, lead_data_original_start.hour, lead_data_original_start.minute, lead_data_original_start.second, lead_data_original_start.microsecond]) 

# and save:
chunk_size = 64 
with h5py.File(output_h5_path, 'a') as f:
    dset_ekg = f.create_dataset(lead, shape=(lead_data.shape[0],), maxshape=(None,),
                              chunks=(chunk_size,), dtype='float64')
    dset_ekg[:] = lead_data.signal.astype('float64')

    dset_startdatetime = f.create_dataset(lead + '_startdatetime', shape=datetime_array.shape, maxshape=(None,),
                              chunks=(chunk_size,), dtype='int64')
    dset_startdatetime[:] = datetime_array.astype('int64')

    dset_fs = f.create_dataset(lead + '_fs', shape=(1,), maxshape=(1,),
                              chunks=(chunk_size,), dtype='int64')
    dset_fs[:] = np.array([new_fs]).astype('int64')